<a href="https://colab.research.google.com/github/Jorge-Ruiz-Troccoli/Data-Science-III/blob/main/transformer_nombres_apellidos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 Extracción Nombre/Apellido T5
Versión optimizada para Colab

## ⚙️ 1. Setup

In [ ]:
import unicodedata
import warnings
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

warnings.filterwarnings("ignore")

MODEL_NAME = "google/flan-t5-small"
PREFIX = "extraer_nombre_apellido: "
OUTPUT_DIR = "mi_modelo_t5"

print("✅ Setup completado")

## 📝 2. Datos

In [ ]:
def strip_accents(text):
    if not isinstance(text, str):
        return text
    nfd = unicodedata.normalize("NFD", text)
    return "".join(c for c in nfd if unicodedata.category(c) != "Mn")

# 📥 Cargar
try:
    df = pd.read_csv("personas.csv", nrows=1000)[["nombre", "apellido"]].dropna()
    df = df.apply(lambda col: col.map(strip_accents) if col.dtype == "object" else col)
    df = df[(df["nombre"].str.len() > 0) & (df["apellido"].str.len() > 0)].reset_index(drop=True)
except FileNotFoundError:
    print("❌ personas.csv no encontrado")
    raise

# 🔄 Generar variantes
target = "nombre: " + df["nombre"] + " apellido: " + df["apellido"]

dataset_df = pd.concat([
    pd.DataFrame({"input": PREFIX + df["nombre"] + " " + df["apellido"], "target": target}),
    pd.DataFrame({"input": PREFIX + df["apellido"] + " " + df["nombre"], "target": target}),
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅ Dataset: {len(dataset_df)} ejemplos")

## 🔧 3. Tokenización (SIN paralelo)

In [ ]:
# 🎲 Split 80/20 (solo train/val, SIN test)
train_df, val_df = train_test_split(dataset_df, test_size=0.2, random_state=42)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#Descarga el tokenizer de flan-t5-small. El tokenizer convierte texto → números (ids), porque el modelo no entiende texto, solo números.

def tokenize(batch):
    inputs = tokenizer(batch["input"], max_length=64, truncation=True, padding=False)
    """
Convierte la columna input (ej: "extraer_nombre_apellido: JORGE RUIZ") a ids. max_length=64 corta si es más largo que eso.
padding=False significa que no rellena con ceros todavía (eso lo hace después el collator, más eficiente).
"""
    labels = tokenizer(text_target=batch["target"], max_length=32, truncation=True, padding=False)
    inputs["labels"] = labels["input_ids"]
    return inputs

"""
Acá tokeniza la salida esperada, target (ej: "nombre: JORGE apellido: RUIZ"). text_target= le dice al tokenizer "esto es la salida,
no la entrada" (algunos modelos tokenizan distinto input vs output).
Esos ids se guardan como labels — es lo que el modelo va a intentar predecir durante el entrenamiento.
Entonces cada fila termina con: input_ids (la pregunta tokenizada) + labels (la respuesta tokenizada).
  """



# ⚡ SIN num_proc (evita overhead)
train_ds = Dataset.from_pandas(train_df, preserve_index=False).map(
    tokenize, batched=True, batch_size=32, remove_columns=["input", "target"]
)
val_ds = Dataset.from_pandas(val_df, preserve_index=False).map(
    tokenize, batched=True, batch_size=32, remove_columns=["input", "target"]
)

"""
Dataset.from_pandas convierte tu DataFrame de pandas a un objeto Dataset (formato que espera la librería datasets,
 más eficiente que pandas para entrenar). .map(tokenize, batched=True, batch_size=32) aplica la función tokenize en lotes de 32 filas a
 la vez (más rápido que fila por fila). remove_columns=["input", "target"] borra las columnas de texto original
  porque ya no las necesitás — el modelo solo usa los ids numéricos.
"""
print(f"✅ Train: {len(train_ds)}, Val: {len(val_ds)}")
print(f"📍 Ejemplo: {tokenizer.decode(train_ds[0]['input_ids'][:10])}...")

## 🚀 4. Entrenamiento (RÁPIDO)

"Seq2Seq" (sequence-to-sequence) es una arquitectura donde el modelo recibe una secuencia de entrada y devuelve otra secuencia de salida, generada token por token, no una clase o un número fijo. Estructura clásica: encoder + decoder.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8)

# ⚡ Config optimizado para velocidad
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,  # ↑ Batch más grande = más rápido
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    learning_rate=3e-4,
    eval_strategy="epoch",  # ✅ Solo al final de cada epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    predict_with_generate=False,  # ⚡ SIN generación en eval (RÁPIDO)
    generation_max_length=32,
    logging_steps=50,  # ↑ Menos logs = menos I/O
    report_to="none",
    gradient_accumulation_steps=1,
)
# 🏋️ Entrenar
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds.select(range(min(1000, len(train_ds)))),
    eval_dataset=val_ds.select(range(min(200, len(val_ds)))),
    data_collator=collator,
    processing_class=tokenizer,
)

print("🏋️ Iniciando entrenamiento...")
history = trainer.train()
print(f"✅ Entrenamiento completado. Loss final: {history.training_loss:.4f}")

## 💾 5. Guardar

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Modelo guardado en {OUTPUT_DIR}")

## 🔮 6. Inferencia (GREEDY = RÁPIDO)

In [ ]:
modelo = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR)
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

def extraer(texto: str) -> str:
    """Extrae nombre y apellido."""
    ids = tokenizer(PREFIX + texto, return_tensors="pt").input_ids
    out = modelo.generate(ids, max_length=32)  # ⚡ Greedy (sin beams)
    return tokenizer.decode(out[0], skip_special_tokens=True)

# 🧪 Pruebas
ejemplos = [
    "JORGE LUIS RUIZ",
    "GARCIA MARIA"
]

print("🧪 Pruebas:\n")
for ej in ejemplos:
    print(f"  📥 {ej:20s} → 🤖 {extraer(ej)}")

In [ ]:
extraer("MARIA DE LOS ANGELES GARCIA")